In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
data = pd.read_csv("spambase_csv.csv")
data.head()

In [ ]:
print("Shape :", data.shape)

print("\nColumns")
print(data.columns)

print("\nMissing Values")
print(data.isnull().sum())

data.info()

In [ ]:
X = data.drop("class", axis=1)
y = data["class"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
data["class"].value_counts().plot(kind="bar")

plt.title("Spam and Ham")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(data.corr(), cmap="coolwarm")
plt.colorbar()

plt.title("Correlation Heatmap")
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training :", X_train.shape)
print("Testing :", X_test.shape)

In [ ]:
logistic = LogisticRegression(max_iter=1000)

start = time.time()
logistic.fit(X_train, y_train)
logistic_time = time.time() - start

logistic_pred = logistic.predict(X_test)

accuracy = accuracy_score(y_test, logistic_pred)
precision = precision_score(y_test, logistic_pred)
recall = recall_score(y_test, logistic_pred)
f1 = f1_score(y_test, logistic_pred)

result = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "Training Time"],
    "Value": [accuracy, precision, recall, f1, logistic_time]
})

result

In [ ]:
parameters = {
    "C": [0.1, 1, 10],
    "solver": ["liblinear"]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=5000),
    parameters,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters :", grid.best_params_)
print("Best Accuracy :", grid.best_score_)

In [ ]:
linear_model = SVC(kernel="linear")

start = time.time()
linear_model.fit(X_train, y_train)
linear_time = time.time() - start

linear_prediction = linear_model.predict(X_test)

In [ ]:
poly_model = SVC(
    kernel="poly",
    degree=3
)

start = time.time()
poly_model.fit(X_train, y_train)
poly_time = time.time() - start

poly_prediction = poly_model.predict(X_test)

In [ ]:
rbf_model = SVC(kernel="rbf")

start = time.time()
rbf_model.fit(X_train, y_train)
rbf_time = time.time() - start

rbf_prediction = rbf_model.predict(X_test)

In [ ]:
sigmoid_model = SVC(kernel="sigmoid")

start = time.time()
sigmoid_model.fit(X_train, y_train)
sigmoid_time = time.time() - start

sigmoid_prediction = sigmoid_model.predict(X_test)

In [ ]:
svm_parameters = {
    "kernel": ["linear", "rbf"],
    "C": [1, 10],
    "gamma": ["scale"]
}

svm_grid = GridSearchCV(
    SVC(),
    svm_parameters,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print("Best Parameters :", svm_grid.best_params_)
print("Best Accuracy :", svm_grid.best_score_)

In [ ]:
logistic_result = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Training Time"
    ],
    "Value": [
        accuracy_score(y_test, logistic_pred),
        precision_score(y_test, logistic_pred),
        recall_score(y_test, logistic_pred),
        f1_score(y_test, logistic_pred),
        logistic_time
    ]
})

logistic_result

In [ ]:
svm_result = pd.DataFrame({
    "Kernel": [
        "Linear",
        "Polynomial",
        "RBF",
        "Sigmoid"
    ],
    "Accuracy": [
        accuracy_score(y_test, linear_prediction),
        accuracy_score(y_test, poly_prediction),
        accuracy_score(y_test, rbf_prediction),
        accuracy_score(y_test, sigmoid_prediction)
    ],
    "F1 Score": [
        f1_score(y_test, linear_prediction),
        f1_score(y_test, poly_prediction),
        f1_score(y_test, rbf_prediction),
        f1_score(y_test, sigmoid_prediction)
    ],
    "Training Time": [
        linear_time,
        poly_time,
        rbf_time,
        sigmoid_time
    ]
})

svm_result

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    logistic_pred
)

plt.title("Logistic Regression")
plt.show()

In [ ]:
best_model = svm_grid.best_estimator_
best_prediction = best_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_prediction
)

plt.title("Best SVM")
plt.show()

In [ ]:
logistic_cv = cross_val_score(
    logistic,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

svm_cv = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

cv_result = pd.DataFrame({
    "Fold": [
        "Fold 1",
        "Fold 2",
        "Fold 3",
        "Fold 4",
        "Fold 5",
        "Average"
    ],
    "Logistic Regression": [
        logistic_cv[0],
        logistic_cv[1],
        logistic_cv[2],
        logistic_cv[3],
        logistic_cv[4],
        logistic_cv.mean()
    ],
    "SVM": [
        svm_cv[0],
        svm_cv[1],
        svm_cv[2],
        svm_cv[3],
        svm_cv[4],
        svm_cv.mean()
    ]
})

cv_result

In [ ]:
tuning_result = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Support Vector Machine"
    ],
    "Search Method": [
        "GridSearchCV",
        "GridSearchCV"
    ],
    "Best Parameters": [
        str(grid.best_params_),
        str(svm_grid.best_params_)
    ],
    "Best CV Accuracy": [
        grid.best_score_,
        svm_grid.best_score_
    ]
})

tuning_result

In [ ]:
comparison = pd.DataFrame({
    "Criterion": [
        "Accuracy",
        "Model Complexity",
        "Training Time",
        "Interpretability"
    ],
    "Logistic Regression": [
        accuracy_score(y_test, logistic_pred),
        "Low",
        logistic_time,
        "High"
    ],
    "SVM": [
        accuracy_score(y_test, best_prediction),
        "High",
        min(linear_time, poly_time, rbf_time, sigmoid_time),
        "Low"
    ]
})

comparison

In [ ]:
models = ["Logistic Regression", "Best SVM"]

scores = [
    accuracy_score(y_test, logistic_pred),
    accuracy_score(y_test, best_prediction)
]

plt.figure(figsize=(5,4))
plt.bar(models, scores)
plt.ylabel("Accuracy")
plt.title("Classifier Comparison")
plt.show()